# LightGCN cross-media training (Kaggle GPU)

Trains + evaluates LightGCN on an Amazon Reviews 2023 category using a free Kaggle **T4 GPU**, then hands you a small `.npz` embeddings file to download and drop into the project's `models/graph/artifacts/`.

**Before Run All:**
1. **Settings -> Accelerator = `GPU T4 x2`** (NOT P100 - Kaggle's PyTorch build dropped support for the P100's sm_60 architecture). **Settings -> Internet = On**. Both need a phone-verified Kaggle account.
2. Make sure your GitHub repo has the latest code. From your laptop:
   ```
   git add models/ scripts/ notebooks/
   git commit -m "Update LightGCN benchmark pipeline"
   git push
   ```
   (Cell 2 pulls the latest `origin/main` on every run, so you never need to re-clone by hand.)
3. Then **Run All**. About 25-40 min at `MAX_USERS = 40000`, or 1.5-2.5 hr at `200000`.

To train the other domains later, change `CATEGORY` in the config cell and Run All again.

**Reading the results:** Amazon data is very sparse (~0.04% density), so absolute numbers are small. Recall@20 around **0.03-0.08** is normal and publishable. What matters is that LightGCN (Step 4) clearly beats the popularity baseline (Step 3).

In [ ]:
# ---------------- CONFIG (edit these) ----------------
REPO_URL   = "https://github.com/sanjay-pokee/Real-Time-Adaptive-Cross-Media-Recommendation-System.git"

CATEGORY   = "Movies_and_TV"   # then rerun with "Books", then "CDs_and_Vinyl"
MAX_USERS  = 200000            # 0 = use every user (slower, ~3-5x). 200k is a large, credible sample.
EPOCHS     = 60
USER_CORE  = 10
ITEM_CORE  = 10
EVAL_K     = 20
EVAL_BATCH = 2048             # fine on 16 GB; lower to 256 only if you hit out-of-memory
# ----------------------------------------------------

ART_NAME = f"lightgcn_amazon_{CATEGORY.lower()}.npz"
print("category:", CATEGORY, "| max_users:", MAX_USERS, "| epochs:", EPOCHS)

In [ ]:
# Clone (or update) the repo, install the two deps Kaggle doesn't ship, check the GPU.
import os, subprocess, sys

WORK = "/kaggle/working"
REPO_DIR = os.path.join(WORK, "repo")
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    # Re-run: pull whatever you last pushed. datasets/ is gitignored, so the
    # already-downloaded CSV survives and does not need re-downloading.
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", "main"], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", "FETCH_HEAD"], check=True)
    print("repo updated to latest origin/main")
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    print("repo cloned")
os.chdir(REPO_DIR)
print("cwd:", os.getcwd())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "huggingface_hub", "pandas"], check=True)

import torch
print("torch:", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "|", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU")
if torch.cuda.is_available() and torch.cuda.get_device_capability(0)[0] < 7:
    print("WARNING: this GPU is too old for Kaggle's PyTorch build. "
          "Switch Settings -> Accelerator to 'GPU T4 x2'.")

# Sanity check: the repo must contain the benchmark-dataset loader.
assert os.path.isfile("models/graph/datasets.py"), (
    "models/graph/datasets.py missing -> your pushed GitHub code is stale. "
    "Commit + push from your laptop, then re-run this cell."
)
print("repo code OK")

## Step 1 - download the category file from HuggingFace

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "scripts.fetch_amazon_dataset", "--category", CATEGORY], check=True)

import glob
matches = glob.glob(f"datasets/amazon/**/{CATEGORY}.csv", recursive=True)
assert matches, "download did not produce a CSV"
DATA_PATH = matches[0]
print("data file:", DATA_PATH)

## Step 2 - preview the filtered graph size

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "models.graph.datasets", DATA_PATH,
                "--user-core", str(USER_CORE), "--item-core", str(ITEM_CORE),
                "--max-users", str(MAX_USERS)], check=True)

## Step 3 - most-popular baseline (the number LightGCN must beat)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "models.graph.evaluate_lightgcn",
                "--dataset", DATA_PATH,
                "--user-core", str(USER_CORE), "--item-core", str(ITEM_CORE),
                "--max-users", str(MAX_USERS),
                "--baseline", "popularity", "--k", str(EVAL_K)], check=True)

## Step 4 - train + evaluate LightGCN (holdout metrics)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "models.graph.evaluate_lightgcn",
                "--dataset", DATA_PATH,
                "--user-core", str(USER_CORE), "--item-core", str(ITEM_CORE),
                "--max-users", str(MAX_USERS),
                "--epochs", str(EPOCHS), "--k", str(EVAL_K),
                "--eval-batch-size", str(EVAL_BATCH)], check=True)

## Step 5 - train the final artifact on ALL kept interactions and save it

In [ ]:
import subprocess, sys, shutil, os
OUT = f"/kaggle/working/{ART_NAME}"
subprocess.run([sys.executable, "-m", "models.graph.train_lightgcn",
                "--dataset", DATA_PATH,
                "--user-core", str(USER_CORE), "--item-core", str(ITEM_CORE),
                "--max-users", str(MAX_USERS),
                "--epochs", str(EPOCHS),
                "--output", OUT], check=True)

for ext in (".npz", ".json"):
    src = OUT[:-4] + ext
    if os.path.isfile(src):
        print("artifact:", src, f"({os.path.getsize(src)/1e6:.1f} MB)")
print("\nDownload these from the Kaggle 'Output' tab, then put the .npz in")
print("your local  models/graph/artifacts/  and point settings.lightgcn_artifact_path at it.")

## Done

- Read off **Recall@20 / NDCG@20 / MRR@20** from Step 4 for LightGCN vs Step 3 for popularity. LightGCN should win on every metric.
- Grab the `.npz` + `.json` from the **Output** tab (right sidebar).
- For the other domains: set `CATEGORY = "Books"` (or `"CDs_and_Vinyl"`) in the config cell, then **Run All** again.